# Part 6 · Notebook 03 — BSM, Black-76 and first-order Greeks

**Sessions:** S3 (BSM, Black-76 & first-order Greeks) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Write the Black–Scholes–Merton price with a dividend yield.
2. Write delta and vega, and check them against the curves.
3. Convert raw Greeks to the units a broker shows.
4. Price an option on a future with Black-76.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()

## 1. The BSM price

With a continuous dividend yield `q`: `V = cp·(S e^{−qT} N(cp·d1) − K e^{−rT} N(cp·d2))`, where `cp = +1` for a call, `−1` for a put, and `p.d1d2` gives `d1 = [ln(S/K) + (r − q + σ²/2)T]/(σ√T)`, `d2 = d1 − σ√T`. `p.N` is the normal CDF. Arrays should work too.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def bsm_price(S, K, T, r, q, sigma, cp):
    d1, d2 = p.d1d2(S, K, T, r, q, sigma)
    return ...                                    # ✍️

K = np.array([540.0, 600.0, 660.0])
mine = [bsm_price(600.0, K, 30 / 365, 0.045, 0.013, 0.18, 1), bsm_price(600.0, K, 30 / 365, 0.045, 0.013, 0.18, -1)]
mine = p.check("bsm_price", mine, [p.bsm_price(600.0, K, 30 / 365, 0.045, 0.013, 0.18, cp) for cp in (1, -1)])
pd.DataFrame({"strike": K, "call": mine[0], "put": mine[1]}).round(4)

In [ ]:
gap = p.parity_gap(mine[0], mine[1], 600.0, K, 30 / 365, 0.045, 0.013)
print("put–call parity residual C − P − (S e^{−qT} − K e^{−rT}):", gap)

## 2. Delta and vega

* `delta = cp · e^{−qT} · N(cp·d1)`
* `vega = S · e^{−qT} · n(d1) · √T` (per 1.00 of σ, the same for calls and puts; `p.n` is the normal density)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def delta_vega(S, K, T, r, q, sigma, cp):
    d1, _ = p.d1d2(S, K, T, r, q, sigma)
    delta = ...                                   # ✍️
    vega = ...                                    # ✍️
    return delta, vega

grid = np.linspace(480, 720, 7)
mine = [delta_vega(grid, 600.0, 30 / 365, 0.045, 0.013, 0.18, cp) for cp in (1, -1)]
ref = [tuple(p.greeks(grid, 600.0, 30 / 365, 0.045, 0.013, 0.18, cp)[g] for g in ("delta", "vega")) for cp in (1, -1)]
mine = p.check("delta and vega", mine, ref)
pd.DataFrame({"S": grid, "call Δ": mine[0][0], "put Δ": mine[1][0], "vega": mine[0][1]}).round(4)

In [ ]:
Sg = np.linspace(450, 750, 300)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for dte, col in [(90, p.PALETTE[0]), (30, p.PALETTE[1]), (5, p.PALETTE[7])]:
    g = p.greeks(Sg, 600.0, dte / 365, 0.045, 0.013, 0.18, 1)
    for ax, name in zip(axes, ("delta", "gamma", "vega", "theta")):
        ax.plot(Sg, g[name], color=col, label=f"{dte} DTE")
for ax, name in zip(axes, ("delta", "gamma", "vega", "theta (per year)")):
    ax.set_title(name); ax.axvline(600, color="#e6e5e0", lw=1)
axes[0].legend(); plt.tight_layout(); plt.show()

## 3. Units: the most common "my Greeks are wrong" bug

The library's raw units are chosen for maths; brokers and py_vollib show **vega per vol point** (÷100), **theta per calendar day** (÷365) and **rho per 1%** (÷100). Delta and gamma don't change. Return a **new** dict; don't modify the input.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def to_display(g):
    out = dict(g)
    out["vega"], out["theta"], out["rho"] = ...   # ✍️ the three conversions
    return out

raw = p.greeks(600.0, 600.0, 30 / 365, 0.045, 0.013, 0.18, 1)
mine = p.attempt(to_display, raw)
mine = p.check("to_display", mine, p.to_display(raw))
pd.DataFrame({"raw": {k: float(v) for k, v in raw.items()}, "display": {k: float(v) for k, v in mine.items()}}).round(4)

Read it as: this call gains about 0.68 if implied vol rises one point, and loses about 0.23 a day with nothing else changing. Compare a raw vega of 68 with a broker's 0.68 and you'd think your model was 100× off.

## 4. Black-76: options on futures

An option on a future has no carry of its own to model: use BSM with `S = F` and `q = r`. Check it against pricing the same option off spot with the carry that produced `F`.

In [ ]:
S, r, q, T, sig = 600.0, 0.045, 0.013, 30 / 365, 0.18
F = p.fair_value(S, r, q, T)
print(f"Black-76 on F = {F:.3f}:     {p.black76_price(F, 610.0, T, r, sig, 1):.6f}")
print(f"BSM on spot with q = {q:.3f}: {p.bsm_price(S, 610.0, T, r, q, sig, 1):.6f}   (same option, same answer)")

## Wrap-up

* One pricer, `cp = ±1`, arrays in and out; Black-76 is BSM with `S = F, q = r`.
* Keep raw units inside the library; convert once, at the display edge.
* Graded version: `labs/part06/week21_pricing_iv` (prices and all first-order Greeks against py_vollib golden values).